<a href="https://colab.research.google.com/github/Saivamshi-K/Mutanews-Beyond-Static-Detection-of-Fake-News/blob/main/Mutanews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

data_path = "/content/drive/MyDrive/Mutanews/data/"

fake_df = pd.read_csv(data_path + "Fake.csv")
true_df = pd.read_csv(data_path + "True.csv")

# Add labels
fake_df["label"] = 0
true_df["label"] = 1

# Combine
df = pd.concat([fake_df, true_df], ignore_index=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset size:", len(df))
df.head()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

print("Using device:", device)

In [ ]:
texts = df["text"].tolist()  # Fake.csv/True.csv usually have 'text'
labels = df["label"].tolist()

encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=128,   # faster than 512
    return_tensors="pt"
)

labels = torch.tensor(labels)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = labels[idx]
        return item

dataset = NewsDataset(encodings, labels)

# 80/20 split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

In [ ]:
from tqdm import tqdm

epochs = 2  # start small

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    print(f"\nAverage Training Loss: {total_loss/len(train_loader)}")

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)

            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    print(f"Validation Accuracy: {correct/total:.4f}")
    print("--------------------------------------------------")

In [ ]:
model.save_pretrained("/content/drive/MyDrive/Mutanews/bert_fake_news_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Mutanews/bert_fake_news_model")

In [ ]:
def predict_news(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    if prediction == 0:
        return "Fake News"
    else:
        return "Real News"


# Test
print(predict_news("The government announced a new policy today."))
print(predict_news("Aliens landed in New York and took control of the White House."))

Fake News
Fake News


In [ ]:
print(predict_news("""
WASHINGTON (Reuters) - The government announced a new economic policy on Tuesday aimed at stabilizing markets.
"""))

Real News


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("Evaluation data collected.")

Evaluation data collected.


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[4646    3]
 [   1 4330]]


In [ ]:
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=["Fake", "Real"]))


Classification Report:

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      4649
        Real       1.00      1.00      1.00      4331

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980



In [ ]:
!pip install transformers accelerate pandas matplotlib networkx

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
gen_model_name = "mistralai/Mistral-7B-Instruct-v0.2"

gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)

gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

In [ ]:
def generate_mutations(news_text, num_mutations=3):

    prompt = f"""
You are a misinformation generator used for research purposes.

Original News:
{news_text}

Create {num_mutations} misleading mutated headlines based on this news.
The mutations should slightly distort the facts but still sound believable.

Return ONLY the mutated headlines as a numbered list.
"""

    # Tokenize input
    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(gen_model.device)

    # Generate text
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=gen_tokenizer.eos_token_id
    )

    # Decode output
    generated_text = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove the prompt part and keep only generated mutations
    mutations = generated_text.replace(prompt, "").strip()

    return mutations

In [ ]:
sample_news = df['text'].sample(5).tolist()

In [ ]:
mutations_data = []

for news in sample_news:

    mutation_output = generate_mutations(news)

    mutations_data.append({
        "original_news": news,
        "mutations": mutation_output
    })

mutations_df = pd.DataFrame(mutations_data)

mutations_df.head()

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
def clean_mutations(text, max_sentences=3):

    sentences = text.split(".")
    cleaned = []

    for s in sentences:
        s = s.strip()

        if len(s) > 20:   # ignore very short fragments
            cleaned.append(s + ".")

        if len(cleaned) == max_sentences:
            break

    return "\n".join(cleaned)

In [ ]:
mutations_df["clean_mutations"] = mutations_df["mutations"].apply(clean_mutations)

In [ ]:
mutations_df[["original_news", "clean_mutations"]]

In [ ]:
mutations_df.to_csv("mutanews_evolution_results.csv", index=False)

In [ ]:

import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()

# Store node types
node_types = {}

for index, row in mutations_df.iterrows():
    original = row["original_news"]
    mutation_text = row["mutations"]

    # Add original node
    G.add_node(original)
    node_types[original] = "original"

    lines = mutation_text.split("\n")

    for m in lines:
        if len(m.strip()) > 10:
            G.add_node(m)
            G.add_edge(original, m)
            node_types[m] = "mutation"

# Layout
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.6, seed=42)

# Node colors based on type
node_colors = [
    "#1f78b4" if node_types[n] == "original" else "#33a02c"
    for n in G.nodes()
]

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_size=500,
    node_color=node_colors,
    alpha=0.9
)

# Draw edges
nx.draw_networkx_edges(
    G, pos,
    edge_color="gray",
    arrows=True,
    arrowstyle="->",
    arrowsize=15
)

# Smart labeling: only label original + few mutations
labels = {}
for node in G.nodes():
    if node_types[node] == "original":
        labels[node] = "ORIGINAL NEWS"
    else:
        # Shorten mutation text
        labels[node] = node[:40] + "..."

nx.draw_networkx_labels(
    G, pos,
    labels=labels,
    font_size=8,
    font_color="black"
)

# Title
plt.title("Mutanews: Misinformation Evolution Graph", fontsize=16)

# Legend
import matplotlib.patches as mpatches
original_patch = mpatches.Patch(color="#1f78b4", label="Original News Source")
mutation_patch = mpatches.Patch(color="#33a02c", label="Mutated / Altered News")

plt.legend(handles=[original_patch, mutation_patch], loc="best")

plt.axis("off")
plt.show()

In [ ]:
!pip install sentence-transformers seaborn

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
all_mutations = []

for index, row in mutations_df.iterrows():

    original = row["original_news"]
    mutation_text = row["mutations"]

    lines = mutation_text.split("\n")

    for m in lines:
        if len(m.strip()) > 10:
            all_mutations.append({
                "original": original,
                "mutation": m.strip()
            })

mutation_list = [x["mutation"] for x in all_mutations]
original_list = [x["original"] for x in all_mutations]

In [ ]:
original_embeddings = model.encode(original_list)
mutation_embeddings = model.encode(mutation_list)

In [ ]:
similarities = []

for i in range(len(mutation_embeddings)):
    sim = cosine_similarity(
        [original_embeddings[i]],
        [mutation_embeddings[i]]
    )[0][0]

    similarities.append(sim)

similarities[:5]

In [ ]:
mutation_strength = [1 - s for s in similarities]

mutation_strength[:5]

In [ ]:
analysis_df = mutations_df.copy()

analysis_df = analysis_df.loc[analysis_df.index.repeat(1)]

analysis_df["similarity"] = similarities[:len(analysis_df)]
analysis_df["mutation_strength"] = mutation_strength[:len(analysis_df)]

analysis_df.head()

In [ ]:
mutation_vectors = model.encode(mutation_list)

similarity_matrix = cosine_similarity(mutation_vectors)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ==============================
# STEP 1: Build text list
# ==============================
all_texts = []

for _, row in mutations_df.iterrows():
    original = row["original_news"]
    all_texts.append(original)

    for m in row["mutations"].split("\n"):
        if len(m.strip()) > 10:
            all_texts.append(m.strip())

# ==============================
# STEP 2: Generate embeddings
# ==============================
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(all_texts)

# ==============================
# STEP 3: Compute similarity
# ==============================
similarity_matrix = cosine_similarity(embeddings)

# ==============================
# STEP 4: Create readable labels
# ==============================
labels = []
for i in range(len(all_texts)):
    if i == 0:
        labels.append("Original")
    else:
        labels.append(f"M{i}")

# ==============================
# STEP 5: Plot heatmap
# ==============================
plt.figure(figsize=(12, 10))

sns.heatmap(
    similarity_matrix,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.5,
    linecolor='gray',
    xticklabels=labels,
    yticklabels=labels,
    cbar_kws={"label": "Semantic Similarity Score"}
)

# Improve readability
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

# Title + context
plt.title(
    "Mutation Similarity Heatmap\n(Higher values = more similar meaning)",
    fontsize=14
)

plt.tight_layout()
plt.show()

# ==============================
# STEP 6: Sanity check
# ==============================
print(f"Texts: {len(all_texts)}")
print(f"Matrix Shape: {similarity_matrix.shape}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# X-axis (mutation steps)
steps = np.arange(1, len(mutation_strength) + 1)

plt.figure(figsize=(12, 7))

# Line plot
plt.plot(
    steps,
    mutation_strength,
    marker='o',
    linewidth=2
)

# Fill area for better visual perception
plt.fill_between(
    steps,
    mutation_strength,
    alpha=0.1
)

# Highlight max drift point
max_idx = np.argmax(mutation_strength)
plt.scatter(
    steps[max_idx],
    mutation_strength[max_idx],
    s=100,
    zorder=3,
    label="Peak Misinformation"
)

# Threshold zones (interpretation bands)
plt.axhspan(0, 0.3, alpha=0.05, label="Low Drift (Minor Changes)")
plt.axhspan(0.3, 0.7, alpha=0.08, label="Moderate Drift")
plt.axhspan(0.7, 1.0, alpha=0.12, label="High Drift (Misinformation)")

# Labels
plt.title(
    "Misinformation Evolution Strength Over Mutation Steps\n(Higher values indicate stronger deviation from original meaning)",
    fontsize=14
)

plt.xlabel("Mutation Step (Progression of Edits)")
plt.ylabel("Mutation Strength (Semantic Drift Score)")

# Ticks
plt.xticks(steps)

# Grid for readability
plt.grid(alpha=0.3)

# Legend
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
analysis_df.to_csv("mutanews_semantic_analysis.csv", index=False)

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ==============================
# STEP 1: Define domains
# ==============================
domains = [
    "politics",
    "sports",
    "cricket",
    "movies",
    "technology",
    "health",
    "finance"
]

model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode domains
domain_embeddings = model.encode(domains)

# Encode mutations
mutation_embeddings = model.encode(mutation_list)

# ==============================
# STEP 2: Assign domain to each mutation
# ==============================
similarity = cosine_similarity(mutation_embeddings, domain_embeddings)

assigned_domains = np.argmax(similarity, axis=1)

topic_df = pd.DataFrame({
    "mutation": mutation_list,
    "domain_id": assigned_domains,
    "domain_name": [domains[i] for i in assigned_domains]
})

# ==============================
# STEP 3: Find cluster head per domain
# ==============================
cluster_heads = {}

for i, domain in enumerate(domains):
    indices = np.where(assigned_domains == i)[0]

    if len(indices) == 0:
        continue

    cluster_embeds = mutation_embeddings[indices]
    centroid = np.mean(cluster_embeds, axis=0)

    sims = cosine_similarity([centroid], cluster_embeds)[0]

    best_idx = indices[np.argmax(sims)]
    cluster_heads[domain] = mutation_list[best_idx]

# ==============================
# STEP 4: Display results
# ==============================
for domain, head in cluster_heads.items():
    print(f"\n=== Domain: {domain.upper()} ===")
    print(f"Cluster Head (Representative News):\n{head}\n")

    print("Sample Mutations:")
    print(
        topic_df[topic_df["domain_name"] == domain]["mutation"]
        .head(3)
        .to_string(index=False)
    )

In [ ]:
topic_df = pd.DataFrame({
    "mutation": mutation_list,
    "topic": topics
})

topic_df.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 7))

# Unique domains
domains = topic_df["domain_name"].unique()

# Assign each domain a numeric y-position
domain_to_y = {domain: i for i, domain in enumerate(domains)}

# Plot each domain separately
for domain in domains:
    indices = topic_df[topic_df["domain_name"] == domain].index.tolist()
    y_vals = [domain_to_y[domain]] * len(indices)

    plt.scatter(
        indices,
        y_vals,
        s=80,
        label=domain.capitalize()
    )

# Y-axis labels as domain names
plt.yticks(
    list(domain_to_y.values()),
    [d.capitalize() for d in domains]
)

# Labels + title
plt.xlabel("Mutation Step (Evolution Timeline)")
plt.ylabel("News Domain")

plt.title(
    "Domain-wise Evolution of Misinformation\n(How news shifts across domains over mutations)",
    fontsize=14
)

# Grid improves readability
plt.grid(alpha=0.3)

# Legend
plt.legend(title="Domains")

plt.tight_layout()
plt.show()

In [ ]:
influence_scores = similarity_matrix.mean(axis=1)

influence_scores[:5]

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# STEP 1: Encode ONLY mutation_list
model = SentenceTransformer("all-MiniLM-L6-v2")
mutation_embeddings = model.encode(mutation_list)

# STEP 2: Compute similarity matrix ONLY for mutations
similarity_matrix = cosine_similarity(mutation_embeddings)

# STEP 3: Compute influence scores
influence_scores = similarity_matrix.mean(axis=1)

# STEP 4: Create DataFrame
influence_df = pd.DataFrame({
    "mutation": mutation_list,
    "influence_score": influence_scores
})

# STEP 5: Sort (optional but useful)
influence_df = influence_df.sort_values(
    by="influence_score",
    ascending=False
)

# STEP 6: Display
print(influence_df.head())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sort data (descending influence)
sorted_indices = np.argsort(influence_scores)[::-1]
sorted_scores = influence_scores[sorted_indices]
sorted_mutations = [mutation_list[i] for i in sorted_indices]

# Short labels (M1, M2...)
labels = [f"M{i+1}" for i in range(len(sorted_scores))]

plt.figure(figsize=(16, 8))

# Color gradient based on influence
colors = plt.cm.viridis(sorted_scores)

bars = plt.bar(
    labels,
    sorted_scores,
    color=colors
)

# Highlight top 3 mutations
for i in range(min(3, len(bars))):
    bars[i].set_edgecolor('black')
    bars[i].set_linewidth(2)

# ✅ Annotate ALL bars (clean way)
for i in range(len(sorted_scores)):
    plt.text(
        i,
        sorted_scores[i] + 0.01,
        f"{sorted_scores[i]:.2f}",
        ha='center',
        fontsize=7,
        rotation=90  # avoids overlap
    )

# Threshold lines
plt.axhline(0.3, linestyle='--', color='orange', alpha=0.7, label="Low Influence Threshold")
plt.axhline(0.6, linestyle='--', color='red', alpha=0.7, label="High Influence Threshold")

# Titles and labels
plt.title(
    "Mutation Influence Scores (Ranked)\n(Higher score = more central and influential misinformation)",
    fontsize=15,
    fontweight='bold'
)

plt.xlabel("Mutations (Ranked)", fontsize=12)
plt.ylabel("Influence Score", fontsize=12)

# Improve x-axis readability
plt.xticks(rotation=45, fontsize=9)

# Grid
plt.grid(axis='y', linestyle='--', alpha=0.4)

# Legend
plt.legend()

# Tight layout for paper fit
plt.tight_layout()

plt.show()

In [ ]:
topic_df.to_csv("mutanews_topics.csv", index=False)

influence_df.to_csv("mutanews_influence_scores.csv", index=False)

In [ ]:
import networkx as nx
tree = nx.DiGraph()

for index, row in mutations_df.iterrows():

    original = row["original_news"]
    mutation_text = row["mutations"]

    lines = mutation_text.split("\n")

    prev_node = original
    tree.add_node(prev_node)

    for m in lines:

        mutation = m.strip()

        if len(mutation) > 10:

            tree.add_node(mutation)

            tree.add_edge(prev_node, mutation)

            prev_node = mutation


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

plt.figure(figsize=(16, 12))

# Layout tuned for tree clarity
pos = nx.spring_layout(tree, k=0.6, seed=42)

# Identify root (node with in-degree 0)
root_nodes = [n for n, d in tree.in_degree() if d == 0]

# Node colors
node_colors = []
for node in tree.nodes():
    if node in root_nodes:
        node_colors.append("#1f78b4")  # blue → original/root
    else:
        node_colors.append("#33a02c")  # green → mutations

# Draw nodes
nx.draw_networkx_nodes(
    tree,
    pos,
    node_color=node_colors,
    node_size=700,
    alpha=0.9
)

# Draw edges (directed)
nx.draw_networkx_edges(
    tree,
    pos,
    edge_color="gray",
    arrows=True,
    arrowstyle="->",
    arrowsize=15,
    width=1.5
)

# Smart labels (shortened text)
labels = {}
for node in tree.nodes():
    if node in root_nodes:
        labels[node] = "ORIGINAL NEWS"
    else:
        labels[node] = node[:35] + "..."

nx.draw_networkx_labels(
    tree,
    pos,
    labels=labels,
    font_size=9
)

# Title
plt.title(
    "Mutanews Evolution Tree: Misinformation Mutation Path\n(Arrows show how original news transforms into misleading versions)",
    fontsize=16
)

# Legend
import matplotlib.patches as mpatches
original_patch = mpatches.Patch(color="#1f78b4", label="Original News (Root)")
mutation_patch = mpatches.Patch(color="#33a02c", label="Mutated News")

plt.legend(handles=[original_patch, mutation_patch], loc="best")

plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# Mutanews Virality Potential Module
# ==============================

import numpy as np
import matplotlib.pyplot as plt

# Convert similarity matrix to numpy array if needed
similarity_matrix = np.array(similarity_matrix)

# ------------------------------
# 1. Mutation Strength
# (How much the mutation deviates from the original)
# ------------------------------
mutation_strength = 1 - similarity_matrix[0]

# ------------------------------
# 2. Mutation Influence Score
# (How central the mutation is among all mutations)
# ------------------------------
influence_scores = similarity_matrix.mean(axis=1)

# ------------------------------
# 3. Narrative Novelty
# (How different it is from the original news)
# ------------------------------
novelty_scores = 1 - similarity_matrix[:, 0]

# ------------------------------
# 4. Virality Potential Score
# ------------------------------
virality_scores = (
    0.4 * mutation_strength +
    0.4 * influence_scores +
    0.2 * novelty_scores
)

# Print results
print("Virality Potential Scores:")
for i, score in enumerate(virality_scores):
    print(f"Mutation {i}: {score:.3f}")

# ------------------------------
# 5. Visualization
# ------------------------------
plt.figure(figsize=(8,5))

plt.bar(range(len(virality_scores)), virality_scores)

plt.title("Mutanews: Virality Potential of Misinformation Mutations")
plt.xlabel("Mutation Index")
plt.ylabel("Virality Score")

plt.show()

In [ ]:
sim_df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)

long_df = sim_df.reset_index().melt(id_vars='index')
long_df.columns = ['Text1', 'Text2', 'Similarity']

long_df.to_csv("similarity_heatmap.csv", index=False)